In [ ]:
__nbid__ = '0080'
__author__ = 'Seo-Won Chang and the KS4 Team'
__version__ = '20260220' # yyyymmdd; version datestamp of this notebook
__datasets__ = ['ks4_dr1']
__keywords__ = ['photometry', 'quality flags', 'color-magnitude diagram', 'survey overview', 'spatial query']

# Getting Started with KS4 DR1
_Seo-Won Chang and the KS4 Team_

### Table of contents
* [Goals & Summary](#goals)
* [Disclaimer & attribution](#attribution)
* [Imports & setup](#import)
* [1. Survey Overview](#overview)
* [2. Exploring the KS4 DR1 Schema](#schema)
  * [2.1 The `idual_master` table](#idual)
  * [2.2 The `single_master` table](#single)
* [3. Basic Queries](#basic)
* [4. Essential Quality Cuts](#quality)
  * [4.1 SExtractor flags](#flags)
  * [4.2 Image flags (imaflags_iso)](#imaflags)
  * [4.3 Spurious source flag (ssflag)](#ssflag)
  * [4.4 Combining all quality cuts](#combined)
* [5. Extinction Correction](#extinction)
* [6. Spatial Queries with Q3C](#spatial)
* [7. Visualizing the Sky with ipyaladin](#aladin)
* [8. Magnitude Distributions & Survey Depth](#depth)
* [9. Color-Magnitude Diagram: 47 Tucanae](#cmd)
* [10. Exploring Dithering Patterns with ndith](#dithering)
* [11. Accessing the Band-Merged Catalog](#singleepoch)
* [Next Steps & Resources](#resources)

<a class="anchor" id="goals"></a>
# Goals
* Learn how to access and query the KS4 DR1 photometric catalogs on Astro Data Lab
* Understand the essential quality parameters and how to apply appropriate filtering
* Perform spatial queries and visualize results
* Generate basic diagnostic plots (magnitude distributions, color-magnitude diagrams)
* Access single-epoch data for time-domain science

# Summary
The KMTNet Synoptic Survey of the Southern Sky (KS4) is a deep, wide-field optical imaging survey covering a southern footprint of $-85^\circ < \mathrm{Decl.} < -28.8^\circ$ in the **B, V, R, and I** bands using a network of three 1.6-m telescopes located in Chile (CTIO), South Africa (SAAO), and Australia (SSO). Although primarily designed to secure reference imaging for gravitational-wave counterpart identification, DR1 delivers science-ready data for $\sim$4,000 deg$^2$ to enable a broad range of astrophysical research.

The release includes deep co-added images reaching median 5$\sigma$ depths of 22.0–23.5 AB mag, accompanied by two source catalogs containing over 200 million sources with SNR > 5:

* **`idual_master`** — an I-band-selected forced-photometry catalog optimized for consistent multi-band colors
* **`single_master`** — a band-merged catalog offering enhanced completeness

Validation demonstrates robust data quality: mean astrometric offsets of +0.054 ± 0.129 arcsec in RA and −0.015 ± 0.120 arcsec in Dec relative to Gaia DR3, with photometric uniformity for point sources maintained within ±0.03 mag relative to Gaia XP for 97.5–99.8% of the footprint across all four bands.

All data products are publicly available via the CDS and NOIRLab's Astro Data Lab. This notebook demonstrates the basic workflow for accessing and using the KS4 DR1 catalogs.

<a class="anchor" id="attribution"></a>
# Disclaimer & attribution
If you use this notebook or KS4 DR1 data for your published science, please acknowledge the following:

* **KS4 Overview paper**: Im et al. (in preparation), *KMTNet Synoptic Survey of Southern Sky I: Overview*, Journal of the Korean Astronomical Society (JKAS)
* **KS4 DR1 pipeline paper**: Jeong et al. (2026), *KMTNet Synoptic Survey of Southern Sky II: Data Reduction and Real-Time Transient Detection Pipeline*, JKAS (accepted for publication)
* **KS4 DR1 paper**: Chang et al. (2026), *KMTNet Synoptic Survey of Southern Sky III: The First Data
Release*, JKAS (in revision)

Acknowledgments
---------------
If you use **Astro Data Lab** in your published research, please include the text in your paper's Acknowledgments section:

_This research uses services or data provided by the Astro Data Lab, which is part of the Community Science and Data Center (CSDC) Program of NSF NOIRLab. NOIRLab is operated by the Association of Universities for Research in Astronomy (AURA), Inc. under a cooperative agreement with the U.S. National Science Foundation._

If you use **SPARCL jointly with the Astro Data Lab platform** (via JupyterLab, command-line, or web interface) in your published research, please include this text below in your paper's Acknowledgments section:

_This research uses services or data provided by the SPectra Analysis and Retrievable Catalog Lab (SPARCL) and the Astro Data Lab, which are both part of the Community Science and Data Center (CSDC) Program of NSF NOIRLab. NOIRLab is operated by the Association of Universities for Research in Astronomy (AURA), Inc. under a cooperative agreement with the U.S. National Science Foundation._

In either case **please cite the following papers**:

* Data Lab concept paper: Fitzpatrick et al., "The NOAO Data Laboratory: a conceptual overview", SPIE, 9149, 2014, https://doi.org/10.1117/12.2057445

* Astro Data Lab overview: Nikutta et al., "Data Lab - A Community Science Platform", Astronomy and Computing, 33, 2020, https://doi.org/10.1016/j.ascom.2020.100411

If you are referring to the Data Lab JupyterLab / Jupyter Notebooks, cite:

* Juneau et al., "Jupyter-Enabled Astrophysical Analysis Using Data-Proximate Computing Platforms", CiSE, 23, 15, 2021, https://doi.org/10.1109/MCSE.2021.3057097

If publishing in a AAS journal, also add the keyword: `\facility{Astro Data Lab}`

And if you are using SPARCL, please also add `\software{SPARCL}` and cite:

* Juneau et al., "SPARCL: SPectra Analysis and Retrievable Catalog Lab", Conference Proceedings for ADASS XXXIII, 2024
https://doi.org/10.48550/arXiv.2401.05576

<a class="anchor" id="import"></a>
# Imports and setup

In [ ]:
# std lib
import warnings
warnings.filterwarnings('ignore')

# 3rd party
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

# Data Lab
from dl import authClient as ac, queryClient as qc, storeClient as sc
from dl.helpers.utils import convert

# Optional: for interactive sky visualization
try:
    import ipyaladin as ipy
    HAS_ALADIN = True
except ImportError:
    HAS_ALADIN = False
    print('ipyaladin not available; sky visualization cells will be skipped.')

# Plotting defaults
plt.rcParams.update({'font.size': 14, 'figure.facecolor': 'white'})

<a class="anchor" id="overview"></a>
# 1. Survey Overview

KS4 DR1 covers $\sim$4,000 deg$^2$ of the southern sky ($-85^\circ < \mathrm{Decl.} < -28.8^\circ$) in four broad-band filters: **B**, **V**, **R**, and **I**. The survey uses the KMTNet telescope system, where each telescope is equipped with a mosaic CCD camera providing a 2° × 2° field of view.

| Property | Value |
|---|---|
| Telescope | KMTNet (CTIO, SAAO, SSO) |
| Aperture | 1.6 m × 3 sites |
| Field of view | 2° × 2° per pointing |
| Filters | B, V, R, I |
| Sky coverage | ~4,000 deg² |
| Declination range | −85° < Dec < −28.8° |
| Median 5σ depth | 22.0–23.5 AB mag |
| Total sources | >200 million (SNR > 5) |
| Photometric system | AB magnitudes (SExtractor) |
| Astrometric reference | Gaia DR3 |
| Photometric reference | Gaia XP |

Let's first look at the sky coverage by querying a random subsample of sources.

In [ ]:
%%time
# Query a random subsample to visualize the survey footprint
# Using the modulo trick on coadd_object_id for efficient random sampling
query_footprint = """
SELECT (healpix_index / 16384) as hpx_coarse,
       AVG(ra) as ra, AVG(dec) as dec,
       AVG(glon) as glon, AVG(glat) as glat,
       COUNT(*) as n_sources
FROM ks4_dr1.idual_master
GROUP BY hpx_coarse
"""
res = qc.query(sql=query_footprint)
df_foot = convert(res, 'pandas')
print(f"Retrieved {len(df_foot):,} sources for footprint visualization.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Equatorial coordinates
ax1.scatter(df_foot['ra'], df_foot['dec'], s=1., alpha=1., c='steelblue', rasterized=True)
ax1.set_xlabel('RA (deg)')
ax1.set_ylabel('Dec (deg)')
ax1.set_title('KS4 DR1 Footprint (Equatorial)')
ax1.invert_xaxis()

# Galactic coordinates
ax2.scatter(df_foot['glon'], df_foot['glat'], s=1., alpha=1., c='firebrick', rasterized=True)
ax2.set_xlabel('Galactic Longitude (deg)')
ax2.set_ylabel('Galactic Latitude (deg)')
ax2.set_title('KS4 DR1 Footprint (Galactic)')

plt.tight_layout()
plt.show()

<a class="anchor" id="schema"></a>
# 2. Exploring the KS4 DR1 Schema

KS4 DR1 provides two main catalog tables:

* **`ks4_dr1.idual_master`**: The I-band-selected forced-photometry catalog, optimized for consistent multi-band colors. Each row represents a unique astronomical source detected in the I-band reference image, with BVRI photometry measured at the I-band detected positions. This is the table most users will start with for studies requiring reliable colors. (~228 million rows)

* **`ks4_dr1.single_master`**: The band-merged catalog offering enhanced completeness. This table includes detections from individual bands that may not appear in the I-band-selected catalog, providing broader source coverage at the cost of less uniform multi-band consistency.

<a class="anchor" id="idual"></a>
## 2.1 The `idual_master` table

The columns follow SExtractor naming conventions with a band suffix (`_b`, `_v`, `_r`, `_i`). Key columns include:

| Column | Description |
|---|---|
| `coadd_object_id` | Unique object identifier |
| `ra`, `dec` | Right Ascension, Declination (J2000, deg) |
| `glon`, `glat` | Galactic longitude, latitude (deg) |
| `ebmv_sfd` | E(B-V) extinction from Schlegel et al. (1998) |
| `mag_auto_{band}` | SExtractor AUTO magnitude |
| `magerr_auto_{band}` | Error in AUTO magnitude |
| `mag_aper{n}_{band}` | Aperture magnitude ({n} arcsec) |
| `class_star_{band}` | SExtractor stellarity index (0=galaxy, 1=star) |
| `fwhm_image_{band}` | FWHM of the object (arcsec) |
| `flags_{band}` | SExtractor flags |
| `imaflags_iso_{band}` | OR-combined flagged pixel types within isophotal radius |
| `ssflag_{band}` | Spurious source flag |
| `sscount_30_{band}` | Spurious source count within 30 arcsec |
| `ndith_{band}` | Number of contributing single-epoch exposures |
| `edge_{band}` | Edge flag |
| `n_det` | Number of detections across all KS4 tiles |
| `healpix_index` | HEALPix index for sky position |

In [ ]:
# Quick peek at the idual_master table
query_peek = """
SELECT coadd_object_id, ra, dec, ebmv_sfd,
       mag_auto_b, magerr_auto_b,
       mag_auto_v, magerr_auto_v,
       mag_auto_r, magerr_auto_r,
       mag_auto_i, magerr_auto_i,
       flags_i, imaflags_iso_i, ssflag_i,
       class_star_i, ndith_i
FROM ks4_dr1.idual_master 
LIMIT 10
"""
res = qc.query(sql=query_peek)
df_peek = convert(res, 'pandas')
df_peek

<a class="anchor" id="single"></a>
## 2.2 The `single_master` table

The `single_master` table is the band-merged catalog, which provides enhanced source completeness by including detections from each band independently. It shares the same column structure as `idual_master`. This table is useful for:

* **Completeness-sensitive studies**: sources detected in one band but not in the I-band reference
* **Band-specific analyses**: when only a single band is needed
* **Comparison with forced photometry**: cross-checking `idual_master` results

In [ ]:
# Quick peek at the single_master table
query_single_peek = """
SELECT coadd_object_id, ra, dec,
       mag_auto_i, magerr_auto_i,
       mean_mjd_i, min_mjd_i, max_mjd_i,
       flags_i, ndith_i
FROM ks4_dr1.single_master
LIMIT 10
"""
res = qc.query(sql=query_single_peek)
df_single_peek = convert(res, 'pandas')
df_single_peek

<a class="anchor" id="basic"></a>
# 3. Basic Queries

Here we demonstrate a few common query patterns. All queries use standard SQL via the Data Lab `queryClient`.

### Count total sources

In [ ]:
%%time
# Count total rows in idual_master
query_count = "SELECT COUNT(*) as total FROM ks4_dr1.idual_master"
res = qc.query(sql=query_count)
df_count = convert(res, 'pandas')
print(f"Total sources in idual_master: {df_count['total'][0]:,}")

### Query bright sources in a specific region

Let's retrieve sources brighter than I = 20 mag within 0.5° of the center of the globular cluster 47 Tucanae (NGC 104).

In [ ]:
%%time
# 47 Tucanae: RA = 6.0236 deg, Dec = -72.0813 deg
query_47tuc = """
SELECT coadd_object_id, ra, dec,
       mag_auto_b, mag_auto_v, mag_auto_r, mag_auto_i,
       magerr_auto_i, class_star_i,
       flags_i, imaflags_iso_i, ssflag_i
FROM ks4_dr1.idual_master
WHERE q3c_radial_query(ra, dec, 6.0236, -72.0813, 0.5)
  AND mag_auto_i < 23
"""
res = qc.query(sql=query_47tuc)
df_47tuc = convert(res, 'pandas')
print(f"Retrieved {len(df_47tuc):,} sources around 47 Tuc with I < 20.")
df_47tuc.head()

<a class="anchor" id="quality"></a>
# 4. Essential Quality Cuts

Before using KS4 photometry for science, it is **critical** to apply appropriate quality filtering. The DR1 catalogs include several flag columns that indicate potential issues with source detection and photometry. Below we describe the three most important flags and the recommended filtering criteria.

> ⚠️ **Warning:** Using KS4 photometry without quality filtering will result in contaminated samples. At minimum, always apply `flags = 0`, `imaflags_iso = 0`, and `ssflag = 0` in the band(s) of interest.

<a class="anchor" id="flags"></a>
## 4.1 SExtractor flags (`flags_{band}`)

The `flags` column is the standard SExtractor extraction flag, a bitmask where each bit indicates a specific issue:

| Bit | Value | Meaning |
|---|---|---|
| 0 | 1 | Object has neighbors, bright enough to significantly bias AUTO photometry |
| 1 | 2 | Object was originally blended with another |
| 2 | 4 | At least one pixel is saturated |
| 3 | 8 | Object is truncated at the image boundary |
| 4 | 16 | Aperture data are incomplete or corrupted |
| 5 | 32 | Isophotal data are incomplete or corrupted |
| 6 | 64 | Memory overflow during deblending |
| 7 | 128 | Memory overflow during extraction |

**Recommended cut: `flags_{band} = 0`** for clean photometry. If you need to retain more sources (e.g., in crowded fields), `flags_{band} <= 3` allows mildly blended objects.

In [ ]:
# Distribution of flags_i for a sample
query_flags = """
SELECT flags_i, COUNT(*) as cnt
FROM ks4_dr1.idual_master
WHERE q3c_radial_query(ra, dec, 6.0236, -72.0813, 0.5)
GROUP BY flags_i
ORDER BY flags_i
"""
res = qc.query(sql=query_flags)
df_flags = convert(res, 'pandas')
print("Distribution of flags_i around 47 Tuc:")
print(df_flags.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(df_flags['flags_i'].astype(str), df_flags['cnt'], color='steelblue', edgecolor='k')
ax.set_xlabel('flags_i')
ax.set_ylabel('Number of sources')
ax.set_title('Distribution of SExtractor flags_i (47 Tuc region)')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

<a class="anchor" id="imaflags"></a>
## 4.2 Image flags (`imaflags_iso_{band}`)

The `imaflags_iso` column is an OR-combined bitmask of flagged pixel types within the isophotal radius of each source. Nonzero values indicate that at least one pixel in the source footprint was flagged (e.g., bad pixel, cosmic ray, saturation bleed trail, etc.).

**Recommended cut: `imaflags_iso_{band} = 0`** for reliable photometry.

> 💡 **Pro Tip:** In crowded fields like globular clusters or the Galactic plane, `imaflags_iso` is especially important for removing sources contaminated by bright neighbor artifacts. Always check this flag in addition to `flags`.

In [ ]:
query_imaflags = """
SELECT imaflags_iso_i, COUNT(*) as cnt
FROM ks4_dr1.idual_master
WHERE q3c_radial_query(ra, dec, 6.0236, -72.0813, 0.5)
GROUP BY imaflags_iso_i
ORDER BY cnt DESC
LIMIT 15
"""
res = qc.query(sql=query_imaflags)
df_imaflags = convert(res, 'pandas')
print("Top 15 imaflags_iso_i values around 47 Tuc:")
print(df_imaflags.to_string(index=False))

<a class="anchor" id="ssflag"></a>
## 4.3 Spurious source flag (`ssflag_{band}`)

The `ssflag` column identifies sources that are likely spurious detections (e.g., diffraction spikes, ghost reflections, satellite trails). Sources with `ssflag != -1` are flagged as potentially spurious.

**Recommended cut: `ssflag_{band} != -1`** to exclude spurious detections.

The companion column `sscount_30_{band}` gives the count of spurious sources within 30 arcsec, which can also be useful for identifying problematic regions around bright stars.

In [ ]:
query_ssflag = """
SELECT ssflag_i, COUNT(*) as cnt
FROM ks4_dr1.idual_master
WHERE q3c_radial_query(ra, dec, 6.0236, -72.0813, 0.5)
GROUP BY ssflag_i
ORDER BY ssflag_i
"""
res = qc.query(sql=query_ssflag)
df_ssflag = convert(res, 'pandas')
print("Distribution of ssflag_i around 47 Tuc:")
print(df_ssflag.to_string(index=False))

<a class="anchor" id="combined"></a>
## 4.4 Combining all quality cuts

For most science applications, the **minimum recommended quality filtering** is:

```sql
WHERE flags_{band} = 0
  AND imaflags_iso_{band} = 0
  AND ssflag_{band} != -1
```

Additionally, you may want to add:
* **Photometric error cut**: `magerr_auto_{band} < 0.1` (or 0.2 for fainter sources)
* **Edge flag**: `edge_{band} = 0` to exclude objects near image boundaries

Let's compare the effect of quality cuts on a sample around 47 Tuc.

In [ ]:
%%time
# Query with full quality cuts applied
query_clean = """
SELECT coadd_object_id, ra, dec,
       mag_auto_b, mag_auto_v, mag_auto_r, mag_auto_i,
       magerr_auto_b, magerr_auto_v, magerr_auto_r, magerr_auto_i,
       class_star_i, ebmv_sfd
FROM ks4_dr1.idual_master
WHERE q3c_radial_query(ra, dec, 6.0236, -72.0813, 0.5)
  AND flags_i = 0
  AND imaflags_iso_i = 0
  AND ssflag_i != -1
  AND mag_auto_i < 23
"""
res = qc.query(sql=query_clean)
df_clean = convert(res, 'pandas')
print(f"Sources after quality cuts: {len(df_clean):,}")
print(f"(Compared to {len(df_47tuc):,} before cuts with I < 20)")

In [ ]:
# Spatial comparison: all sources vs clean sources
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 20))

ax1.scatter(df_47tuc['ra'], df_47tuc['dec'], s=0.3, alpha=0.3, c='grey', rasterized=True)
ax1.set_xlabel('RA (deg)')
ax1.set_ylabel('Dec (deg)')
ax1.set_title(f'All sources (N={len(df_47tuc):,}, I<23)')
ax1.set_aspect('equal')
ax1.invert_xaxis()

ax2.scatter(df_clean['ra'], df_clean['dec'], s=0.3, alpha=0.3, c='steelblue', rasterized=True)
ax2.set_xlabel('RA (deg)')
ax2.set_ylabel('Dec (deg)')
ax2.set_title(f'Clean sources (N={len(df_clean):,}, I<23, all flags=0)')
ax2.set_aspect('equal')
ax2.invert_xaxis()

plt.tight_layout()
plt.show()

<a class="anchor" id="extinction"></a>
# 5. Extinction Correction

Each source in the KS4 DR1 catalog has an associated E(B-V) value from the Schlegel et al. (1998) dust map, stored in the `ebmv_sfd` column. To obtain extinction-corrected magnitudes, use the standard extinction coefficients:

$$A_B = R_B \times E(B-V), \quad A_V = R_V \times E(B-V), \quad A_R = R_R \times E(B-V), \quad A_I = R_I \times E(B-V)$$

where $R_B \approx 3.626$, $R_V \approx 2.742$, $R_R \approx 2.169$, $R_I \approx 1.505$ (values depend on the exact filter transmission; see Schlafly & Finkbeiner (2011) for details).

> ⚠️ **Warning:** The Schlegel et al. (1998) E(B-V) values are known to overestimate extinction in regions of high dust column density ($E(B-V) > 0.5$ mag). Consider using updated maps (e.g., Schlafly & Finkbeiner 2011) for heavily reddened fields.

In [ ]:
# Apply extinction correction to the clean 47 Tuc sample
# Approximate extinction coefficients for KMTNet BVRI filters
R_B, R_V, R_R, R_I = 3.626, 2.742, 2.169, 1.505

df_clean['mag_auto_b_dered'] = df_clean['mag_auto_b'] - R_B * df_clean['ebmv_sfd']
df_clean['mag_auto_v_dered'] = df_clean['mag_auto_v'] - R_V * df_clean['ebmv_sfd']
df_clean['mag_auto_r_dered'] = df_clean['mag_auto_r'] - R_R * df_clean['ebmv_sfd']
df_clean['mag_auto_i_dered'] = df_clean['mag_auto_i'] - R_I * df_clean['ebmv_sfd']

print("Extinction correction applied.")
print(f"Median E(B-V) in this field: {df_clean['ebmv_sfd'].median():.4f} mag")
print(f"Median A_I correction: {(R_I * df_clean['ebmv_sfd']).median():.4f} mag")

In [ ]:
# Compare observed vs dereddened CMD
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 7), sharey=True)

# Observed
color_obs = df_clean['mag_auto_v'] - df_clean['mag_auto_i']
mag_obs = df_clean['mag_auto_i']
ax1.scatter(color_obs, mag_obs, s=0.3, alpha=0.4, c='grey', rasterized=True)
ax1.set_xlabel('V - I (observed)')
ax1.set_ylabel('I (mag)')
ax1.set_title('Observed')
ax1.set_xlim(-0.5, 3.5)
ax1.set_ylim(23, 14.5)
ax1.grid(True, linestyle='--', alpha=0.5)

# Dereddened
color_dered = df_clean['mag_auto_v_dered'] - df_clean['mag_auto_i_dered']
mag_dered = df_clean['mag_auto_i_dered']
ax2.scatter(color_dered, mag_dered, s=0.3, alpha=0.4, c='steelblue', rasterized=True)
ax2.set_xlabel('(V - I)₀ (dereddened)')
ax2.set_title('Dereddened')
ax2.set_xlim(-0.5, 3.5)
ax2.set_ylim(23, 14.5)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.suptitle('47 Tucanae: CMD Before and After Extinction Correction', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

<a class="anchor" id="spatial"></a>
# 6. Spatial Queries with Q3C

The KS4 DR1 tables are indexed with [Q3C](https://github.com/segasai/q3c), a PostgreSQL extension for fast spatial queries. Two key functions are available:

* **`q3c_radial_query(ra, dec, center_ra, center_dec, radius_deg)`** — cone search
* **`q3c_poly_query(ra, dec, '{ra1,dec1,ra2,dec2,...}')`** — polygon search

### Cone search example
Retrieve all sources within 2.0° of the Small Magellanic Cloud center:

In [ ]:
%%time
# SMC center: RA ~ 13.187 deg, Dec ~ -72.829 deg
query_smc = """
SELECT ra, dec, mag_auto_i, mag_auto_v,
       class_star_i, flags_i
FROM ks4_dr1.idual_master
WHERE q3c_radial_query(ra, dec, 13.187, -72.829, 2.)
  AND flags_i = 0
  AND imaflags_iso_i = 0
  AND ssflag_i != -1
"""
res = qc.query(sql=query_smc)
df_smc = convert(res, 'pandas')
print(f"Sources in SMC cone search (r=0.1°): {len(df_smc):,}")

### Box (RA/Dec range) search
For rectangular regions, a simple `BETWEEN` clause can be more efficient than `q3c_poly_query`:

In [ ]:
%%time
# A 0.5 x 0.5 deg box around a sparse field
query_box = """
SELECT ra, dec, mag_auto_i, mag_auto_v, mag_auto_r, mag_auto_b,
       class_star_i
FROM ks4_dr1.idual_master
WHERE ra BETWEEN 49.5 AND 50.5
  AND dec BETWEEN -60.0 AND -59.5
  AND flags_i = 0
  AND imaflags_iso_i = 0
  AND ssflag_i != -1
"""
res = qc.query(sql=query_box)
df_box = convert(res, 'pandas')
print(f"Sources in box search: {len(df_box):,}")

<a class="anchor" id="aladin"></a>
# 7. Visualizing the Sky with ipyaladin

The KS4 DR1 has a HiPS (Hierarchical Progressive Survey) color composite image available, which can be visualized interactively using `ipyaladin`.

In [ ]:
if HAS_ALADIN:
    # Initialize Aladin widget centered on 47 Tucanae
    aladin = ipy.Aladin(target='00 24 05.67 -72 04 52.6', fov=1, height=600)
    display(aladin)

In [ ]:
from ipyaladin import Aladin

aladin = Aladin(
    target='00 24 05.67 -72 04 52.6',
    fov=1,
    height=600,
    survey='https://alasky.cds.unistra.fr/KS4/DR1/SNU_P_KS4_DR1_colorBVI/'
)
display(aladin)

aladin.survey = 'https://alasky.cds.unistra.fr/KS4/DR1/SNU_P_KS4_DR1_colorBVI/'

<a class="anchor" id="depth"></a>
# 8. Magnitude Distributions & Survey Depth

Let's examine the magnitude distributions in each band to understand the survey depth. We query a representative field away from the Galactic plane.

In [ ]:
%%time
# Query a sparse field for magnitude distributions
query_depth = """
SELECT mag_auto_b, mag_auto_v, mag_auto_r, mag_auto_i,
       magerr_auto_b, magerr_auto_v, magerr_auto_r, magerr_auto_i
FROM ks4_dr1.single_master
WHERE q3c_radial_query(ra, dec, 50.0, -60.0, 0.5)
  AND flags_i = 0
  AND imaflags_iso_i = 0
  AND ssflag_i != -1
"""
res = qc.query(sql=query_depth)
df_depth = convert(res, 'pandas')
print(f"Retrieved {len(df_depth):,} clean sources for depth analysis.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Magnitude distributions
bins = np.arange(14, 27, 0.2)
for band, color, label in [('b', 'blue', 'B'), ('v', 'green', 'V'), ('r', 'orange', 'R'), ('i', 'red', 'I')]:
    mag = df_depth[f'mag_auto_{band}'].dropna()
    ax1.hist(mag, bins=bins, histtype='step', lw=2, color=color, label=label, alpha=0.8)

ax1.set_xlabel('Magnitude (AB)')
ax1.set_ylabel('Number of sources')
ax1.set_title('Magnitude Distributions (clean sources)')
ax1.legend(fontsize=12)
ax1.set_yscale('log')

# Photometric error vs magnitude
for band, color, label in [('b', 'blue', 'B'), ('v', 'green', 'V'), ('r', 'orange', 'R'), ('i', 'red', 'I')]:
    mag = df_depth[f'mag_auto_{band}']
    err = df_depth[f'magerr_auto_{band}']
    mask = (mag > 14) & (mag < 26) & (err < 1.0)
    ax2.scatter(mag[mask], err[mask], s=0.1, alpha=0.1, c=color, label=label, rasterized=True)

ax2.axhline(y=0.1, color='k', ls='--', alpha=0.5, label='0.1 mag')
ax2.set_xlabel('Magnitude (AB)')
ax2.set_ylabel('Magnitude error')
ax2.set_title('Photometric Error vs Magnitude')
ax2.set_ylim(0, 0.5)
ax2.legend(fontsize=12, markerscale=20)

plt.tight_layout()
plt.show()

<a class="anchor" id="cmd"></a>
# 9. Color-Magnitude Diagram: 47 Tucanae

One of the most common uses of multi-band photometry is constructing color-magnitude diagrams (CMDs). Here we use the clean, dereddened data from Section 5 to produce CMDs of the globular cluster 47 Tucanae.

In [ ]:
# Use the dereddened clean 47 Tuc sample
# Filter for valid magnitudes in all bands
mask = (
    df_clean['mag_auto_b_dered'].between(10, 28) &
    df_clean['mag_auto_v_dered'].between(10, 28) &
    df_clean['mag_auto_r_dered'].between(10, 28) &
    df_clean['mag_auto_i_dered'].between(10, 28) &
    (df_clean['magerr_auto_i'] < 0.2)
)
df_cmd = df_clean[mask].copy()
print(f"Sources for CMD: {len(df_cmd):,}")

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(22, 7), sharey=True)

# V-I vs I
ax1.scatter(df_cmd['mag_auto_v_dered'] - df_cmd['mag_auto_i_dered'],
            df_cmd['mag_auto_i_dered'],
            s=0.3, alpha=0.15, c='k', rasterized=True)
ax1.set_xlabel('$(V - I)_0$')
ax1.set_ylabel('$I_0$')
ax1.set_title('V - I vs I')
ax1.set_xlim(-0.5, 3.0)
ax1.set_ylim(23, 14.5)

# B-V vs V
ax2.scatter(df_cmd['mag_auto_b_dered'] - df_cmd['mag_auto_v_dered'],
            df_cmd['mag_auto_v_dered'],
            s=0.3, alpha=0.15, c='k', rasterized=True)
ax2.set_xlabel('$(B - V)_0$')
ax2.set_title('B - V vs V')
ax2.set_xlim(-0.5, 2.5)

# R-I vs I
ax3.scatter(df_cmd['mag_auto_r_dered'] - df_cmd['mag_auto_i_dered'],
            df_cmd['mag_auto_i_dered'],
            s=0.3, alpha=0.15, c='k', rasterized=True)
ax3.set_xlabel('$(R - I)_0$')
ax3.set_title('R - I vs I')
ax3.set_xlim(-0.5, 2.0)

# B-I vs I
ax4.scatter(df_cmd['mag_auto_b_dered'] - df_cmd['mag_auto_i_dered'],
            df_cmd['mag_auto_i_dered'],
            s=0.3, alpha=0.15, c='k', rasterized=True)
ax4.set_xlabel('$(B - I)_0$')
ax4.set_title('B - I vs I')
ax4.set_xlim(-0.5, 4.5)

plt.suptitle('47 Tucanae — Color-Magnitude Diagrams (KS4 DR1, dereddened)', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Hess diagram (2D histogram) for a cleaner view of the CMD features
fig, ax = plt.subplots(figsize=(8, 8))

color = df_cmd['mag_auto_v_dered'] - df_cmd['mag_auto_i_dered']
mag = df_cmd['mag_auto_i_dered']

hb = ax.hexbin(color, mag, gridsize=200, bins='log', mincnt=1,
               extent=[-0.5, 3.0, 23, 12], cmap='inferno')
ax.set_xlabel('$(V - I)_0$', fontsize=14)
ax.set_ylabel('$I_0$', fontsize=14)
ax.set_title('47 Tucanae — Hess Diagram', fontsize=15)
plt.colorbar(hb, ax=ax, label='log(N)')
plt.tight_layout()
ax.invert_yaxis()
plt.show()

<a class="anchor" id="dithering"></a>
# 10. Exploring Dithering Patterns with `ndith`

The `ndith_{band}` column indicates the number of single-epoch exposures that contributed to each source's coadded photometry in a given band. This number depends on the dithering strategy and the position within the field of view. Mapping `ndith` spatially reveals the dithering pattern and overlap regions between pointings.

> 💡 **Pro Tip:** Regions with higher `ndith` values have deeper effective exposures and consequently better photometric precision. Use `ndith` to identify the deepest regions or to understand spatial variations in survey depth.

In [ ]:
%%time
# Query ndith values in a field to visualize the dithering pattern
query_ndith = """
SELECT ra, dec, ndith_b, ndith_v, ndith_r, ndith_i
FROM ks4_dr1.idual_master
WHERE q3c_radial_query(ra, dec, 50.0, -60.0, 1.2)
  AND flags_i = 0
"""
res = qc.query(sql=query_ndith)
df_ndith = convert(res, 'pandas')
print(f"Retrieved {len(df_ndith):,} sources before filtering.")

# Filter out null values (9999999) and unreasonable values (ndith > 20)
for band in ['b', 'v', 'r', 'i']:
    col = f'ndith_{band}'
    df_ndith.loc[df_ndith[col] >= 9999999, col] = np.nan
    df_ndith.loc[df_ndith[col] > 20, col] = np.nan

df_ndith = df_ndith.dropna(subset=['ndith_i'])
print(f"After filtering (ndith <= 20, excluding nulls): {len(df_ndith):,} sources.")
print(f"\nndith_i statistics:")
print(df_ndith['ndith_i'].describe())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

for ax, band, label in zip(axes, ['b', 'v', 'r', 'i'], ['B', 'V', 'R', 'I']):
    sc = ax.scatter(df_ndith['ra'], df_ndith['dec'],
                    c=df_ndith[f'ndith_{band}'], s=0.1, alpha=0.5,
                    cmap='viridis', vmin=0, rasterized=True)
    ax.set_xlabel('RA (deg)')
    ax.set_ylabel('Dec (deg)')
    ax.set_title(f'ndith_{label}')
    ax.set_aspect('equal')
    ax.invert_xaxis()
    plt.colorbar(sc, ax=ax, label=f'N dithers ({label})')

plt.suptitle('Dithering Pattern — Number of Contributing Exposures', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

<a class="anchor" id="singleepoch"></a>
# 11. Accessing the Band-Merged Catalog (`single_master`)

The `single_master` table is the band-merged catalog, which provides enhanced completeness by including detections from each band independently, rather than relying solely on I-band detections. This table is particularly useful when you need sources that may be bright in one band but faint or undetected in the I-band reference.

### Comparing the two catalogs

Let's query both catalogs for the same region to see the difference in source counts:

In [ ]:
# Count sources in both catalogs for the same region
query_compare_idual = """
SELECT COUNT(*) as n_idual
FROM ks4_dr1.idual_master
WHERE q3c_radial_query(ra, dec, 6.0236, -72.0813, 0.5)
  AND flags_i = 0
  AND imaflags_iso_i = 0
  AND ssflag_i != -1
"""
res = qc.query(sql=query_compare_idual)
df_idual_count = convert(res, 'pandas')

query_compare_single = """
SELECT COUNT(*) as n_single
FROM ks4_dr1.single_master
WHERE q3c_radial_query(ra, dec, 6.0236, -72.0813, 0.5)
  AND flags_i = 0
  AND imaflags_iso_i = 0
  AND ssflag_i != -1
"""
res = qc.query(sql=query_compare_single)
df_single_count = convert(res, 'pandas')

print(f"Sources within 0.5° of 47 Tuc center (clean):")
print(f"  idual_master (I-band selected):  {df_idual_count['n_idual'][0]:,}")
print(f"  single_master (band-merged):     {df_single_count['n_single'][0]:,}")

In [ ]:
%%time
# Query the band-merged catalog for a sample region
query_single_sample = """
SELECT coadd_object_id, ra, dec,
       mag_auto_b, mag_auto_v, mag_auto_r, mag_auto_i,
       magerr_auto_i, class_star_i,
       flags_i, imaflags_iso_i, ssflag_i, ndith_i
FROM ks4_dr1.single_master
WHERE q3c_radial_query(ra, dec, 6.0236, -72.0813, 0.5)
  AND flags_i = 0
  AND imaflags_iso_i = 0
  AND ssflag_i != -1
"""
res = qc.query(sql=query_single_sample)
df_single_sample = convert(res, 'pandas')
print(f"Retrieved {len(df_single_sample):,} sources from single_master.")
df_single_sample.head()

In [ ]:
# Compare CMD from idual_master vs single_master
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

# idual_master CMD (from Section 9)
mask_idual = (
    df_clean['mag_auto_v'].between(10, 28) &
    df_clean['mag_auto_i'].between(10, 28) &
    (df_clean['magerr_auto_i'] < 0.2)
)
ax1.scatter(df_clean.loc[mask_idual, 'mag_auto_v'] - df_clean.loc[mask_idual, 'mag_auto_i'],
            df_clean.loc[mask_idual, 'mag_auto_i'],
            s=0.3, alpha=0.15, c='k', rasterized=True)
ax1.set_xlabel('V - I')
ax1.set_ylabel('I (mag)')
ax1.set_title(f'idual_master (I-band selected, N={mask_idual.sum():,})')
ax1.set_xlim(-0.5, 3.0)
ax1.set_ylim(23, 14.5)

# single_master CMD
mask_single = (
    df_single_sample['mag_auto_v'].between(10, 28) &
    df_single_sample['mag_auto_i'].between(10, 28) &
    (df_single_sample['magerr_auto_i'] < 0.2)
)
ax2.scatter(df_single_sample.loc[mask_single, 'mag_auto_v'] - df_single_sample.loc[mask_single, 'mag_auto_i'],
            df_single_sample.loc[mask_single, 'mag_auto_i'],
            s=0.3, alpha=0.15, c='k', rasterized=True)
ax2.set_xlabel('V - I')
ax2.set_title(f'single_master (band-merged, N={mask_single.sum():,})')
ax2.set_xlim(-0.5, 3.0)

plt.suptitle('47 Tucanae — Catalog Comparison', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

<a class="anchor" id="resources"></a>
# Next Steps & Resources

This notebook covered the basics of accessing KS4 DR1 data. For more advanced use cases, see the companion notebooks:

* **Notebook 2: Star-Galaxy Separation & X-match** — `class_star` analysis, cross-matching with Gaia DR3 and other surveys

### Useful links
* [KS4 DR1 Paper](https://ui.adsabs.harvard.edu/) (Chang et al. 2026)
* [KS4 DR1 Pipeline Paper](https://ui.adsabs.harvard.edu/) (Jeong et al. 2026)
* [Astro Data Lab](https://datalab.noirlab.edu/)
* [Data Lab Query Interface](https://datalab.noirlab.edu/query.php)
* [Q3C Documentation](https://github.com/segasai/q3c)
* [SExtractor Documentation](https://sextractor.readthedocs.io/)
* [KS4 DR1 HiPS Viewer](https://alasky.cds.unistra.fr/KS4/DR1/SNU_P_KS4_DR1_colorBVI/)